# Этап 3, версия 1.5. Влияет ли ограничение цен на качество модели?

Проверяем идею: оставить объявления с ценой строго между 1-м и 90-м процентилями, а затем разбить их на train/validation/test. Сравниваем две модели LightGBM на **одних и тех же объектах validation**. Test до окончательного выбора не используем. Метрики считаем в TRY, не на логарифмах.

Этот эксперимент меняет область применения: от всех продаж к ценовому сегменту. Он не показывает качество на рынке в целом. Цена объекта заранее неизвестна, поэтому ценовой диапазон нельзя использовать как входной фильтр будущего приложения.


In [ ]:
# Получаем продажи в TRY и функцию создания признаков из этапа 2.
import json
from pathlib import Path
from IPython.utils.capture import capture_output
stage2 = next((p / "notebooks" / "02_preprocessing_v1.7.ipynb"
               for p in (Path.cwd(), Path.cwd().parent)
               if (p / "notebooks" / "02_preprocessing_v1.7.ipynb").exists()), None)
if stage2 is None:
    raise FileNotFoundError("Не найден этап 2 v1.7")
with capture_output():
    for cell in json.loads(stage2.read_text(encoding="utf-8"))["cells"]:
        if cell["cell_type"] == "code":
            result = get_ipython().run_cell("".join(cell["source"]))
            if result.error_before_exec or result.error_in_exec:
                raise result.error_before_exec or result.error_in_exec
print("Продаж после исключения проверенной аномалии:", len(sale))


## 1. Границы и новое разбиение

Перцентили считаем по всем положительным ценам продаж после перевода в TRY и исключения записи № 392157. Не смешиваем продажи с арендой и цены в разных валютах. Для точного повторения предложенного фильтра используем строгие неравенства `>` и `<`. Это **исследовательский отбор по целевой цене**, а не способ подготовить модель для прогноза любого объекта.


In [ ]:
# Сначала фильтруем цены, затем делим отобранные объявления по группам.
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit

q01, q90 = sale.price.quantile([.01, .90])
selected = sale.loc[sale.price.gt(q01) & sale.price.lt(q90)].copy()
group_cols = ["address", "sub_type", "room_count", "size"]
group_selected = pd.util.hash_pandas_object(selected[group_cols], index=False)
first = GroupShuffleSplit(n_splits=1, test_size=.15, random_state=42)
dev_idx, test_idx = next(first.split(selected, groups=group_selected))
dev, range_test = selected.iloc[dev_idx], selected.iloc[test_idx]
second = GroupShuffleSplit(n_splits=1, test_size=.15/.85, random_state=43)
train_idx, val_idx = next(second.split(dev, groups=group_selected.iloc[dev_idx]))
range_train, range_val = dev.iloc[train_idx], dev.iloc[val_idx]

# Полная модель видит и цены вне диапазона, но ни одну группу validation/test.
holdout_groups = set(pd.util.hash_pandas_object(
    pd.concat([range_val, range_test])[group_cols], index=False))
group_all = pd.util.hash_pandas_object(sale[group_cols], index=False)
full_train = sale.loc[~group_all.isin(holdout_groups)].copy()
assert set(range_train.id).issubset(set(full_train.id))
assert set(range_train.id).isdisjoint(set(range_val.id))
assert set(range_train.id).isdisjoint(set(range_test.id))
assert set(range_val.id).isdisjoint(set(range_test.id))
assert set(group_all.loc[full_train.index]).isdisjoint(holdout_groups)
display(pd.DataFrame({
    "Набор": ["Все продажи", "Диапазон 1–90%", "Train диапазона", "Validation диапазона", "Test диапазона", "Train полной модели"],
    "Объявлений": [len(sale), len(selected), len(range_train), len(range_val), len(range_test), len(full_train)]
}))
print(f"Границы: {q01:,.0f} < price < {q90:,.0f} TRY")
print(f"Исключено из диапазона: {len(sale)-len(selected):,} объявлений ({100*(1-len(selected)/len(sale)):.1f}%)")


## 2. Одинаковая подготовка и обучение

Признаки создаются одинаково. Недостающую площадь заполняем по микрорайону и типу жилья, затем по району, типу и общей медиане. Эти медианы, границы площади, кодирование категорий и числовые заполнители вычисляются **отдельно по train каждой модели**. Алгоритм и его настройки совпадают; отличается только набор тренировочных строк.


In [ ]:
# Обучаем две модели; validation и test не участвуют в fit.
import warnings
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from lightgbm import LGBMRegressor

def make_features(train_part, val_part):
    x_train, x_val = features(train_part), features(val_part)
    by_neighborhood = x_train.groupby(["neighborhood", "sub_type"]).size_m2.agg(["median", "count"])
    by_neighborhood = by_neighborhood.loc[by_neighborhood["count"].ge(20), "median"]
    by_district = x_train.groupby(["district", "sub_type"]).size_m2.agg(["median", "count"])
    by_district = by_district.loc[by_district["count"].ge(20), "median"]
    by_type = x_train.groupby("sub_type").size_m2.median()
    overall = x_train.size_m2.median()
    def fill_area(x):
        x = x.copy()
        for keys, medians in [(["neighborhood", "sub_type"], by_neighborhood),
                              (["district", "sub_type"], by_district)]:
            idx = pd.MultiIndex.from_frame(x[keys])
            local = pd.Series(medians.reindex(idx).to_numpy(), index=x.index)
            x["size_m2"] = x.size_m2.fillna(local)
        x["size_m2"] = x.size_m2.fillna(x.sub_type.map(by_type)).fillna(overall)
        return x
    x_train, x_val = fill_area(x_train), fill_area(x_val)
    low, high = x_train.size_m2.quantile([.001, .999])
    for x in (x_train, x_val):
        x["size_m2"] = x.size_m2.clip(low, high)
    return x_train, x_val

def fit_model(train_part):
    x_train, x_val = make_features(train_part, range_val)
    encoder = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num),
        ("cat", Pipeline([
            ("fill", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50))
        ]), cat)
    ], sparse_threshold=.8)
    reg = TransformedTargetRegressor(
        regressor=LGBMRegressor(random_state=42, verbosity=-1, n_jobs=4, num_leaves=15),
        func=np.log1p, inverse_func=np.expm1)
    model = Pipeline([("encode", encoder), ("model", reg)])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model.fit(x_train, train_part.price)
        pred = model.predict(x_val)
    return model, pred

full_model, full_pred = fit_model(full_train)
range_model, range_pred = fit_model(range_train)
print("Обе модели обучены. Validation:", len(range_val), "объявлений; test не использован.")


## 3. Метрики на одних и тех же объектах

MAE, RMSE и MedAE выражены в TRY; MAPE — в процентах. Основной вопрос: уменьшилась ли ошибка на одной и той же validation-выборке. R² для этой выборки нельзя напрямую сравнивать с R² на полном рынке: разброс целевой цены другой.


In [ ]:
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             median_absolute_error, mean_absolute_percentage_error,
                             r2_score)
y = range_val.price.to_numpy()
rows = []
for name, train_part, pred in [
    ("Все цены → диапазон", full_train, full_pred),
    ("Только диапазон → диапазон", range_train, range_pred),
]:
    rows.append({
        "Модель": name, "Train": len(train_part), "Validation": len(y),
        "MAE, TRY": mean_absolute_error(y, pred),
        "RMSE, TRY": np.sqrt(mean_squared_error(y, pred)),
        "MedAE, TRY": median_absolute_error(y, pred),
        "MAPE, %": 100 * mean_absolute_percentage_error(y, pred),
        "R²": r2_score(y, pred),
    })
metrics = pd.DataFrame(rows)
pd.set_option("display.max_columns", None)
display(metrics.round(3))
delta = np.abs(y-full_pred) - np.abs(y-range_pred)
print(f"Разница MAE (полная минус ограниченная): {delta.mean():,.2f} TRY")
print("Положительная разница означает преимущество модели диапазона.")


## 4. Что показывает эксперимент

На **одинаковых 40 073 объявлениях validation** модель, обученная на всех ценах, получила MAE 76 047 TRY, RMSE 168 604 TRY, MAPE 27,28% и R² = −0,410. Модель, обученная только внутри диапазона, получила MAE 64 249 TRY, RMSE 91 419 TRY, MAPE 23,06% и R² = 0,585. Средняя абсолютная ошибка уменьшилась на **11 798 TRY**.

Идея ограничения цен действительно улучшила прогноз **внутри выбранного диапазона**. Но из рассмотрения исключены 32 089 продаж (11,2%), а цена нового объекта заранее неизвестна. Поэтому R² = 0,585 нельзя выдавать за качество модели на всех продажах или напрямую сравнивать со старым R² на другом составе validation. Для приложения нужна оценка на неотфильтрованных объектах или определение сегмента по признакам, доступным до прогноза.

Ценовые границы вычислены на всём наборе до разбиения по условию эксперимента. Это исследовательская проверка с выбором области по целевой переменной, а не строгая оценка качества на будущем потоке всех объявлений. Финальный test оставлен нетронутым.
